In [1]:
import requests
import os
from dotenv import load_dotenv
from newspaper import Article
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

load_dotenv()

API_KEY = os.getenv("ZENSERP_API_KEY")


# --------------------------------
# Search with Zenserp
# --------------------------------
def search_google(query):

    url = "https://app.zenserp.com/api/v2/search"

    params = {
        "q": query,
        "gl": "us",
        "num": 20
    }

    headers = {"apikey": API_KEY}

    response = requests.get(url, headers=headers, params=params)

    return response.json()


# --------------------------------
# Extract article text
# --------------------------------
def extract_article(url):

    try:
        article = Article(url)
        article.download()
        article.parse()

        return article.title, article.text

    except:
        return None, None


# --------------------------------
# Create PDF
# --------------------------------
def create_pdf(filename, sections):

    styles = getSampleStyleSheet()
    story = []

    for section, articles in sections.items():

        story.append(Paragraph(section, styles["Title"]))
        story.append(Spacer(1, 20))

        for title, text in articles:

            story.append(Paragraph(title, styles["Heading2"]))
            story.append(Spacer(1, 10))

            paragraphs = text.split("\n")

            for p in paragraphs:

                if len(p.strip()) > 120:
                    story.append(Paragraph(p, styles["BodyText"]))
                    story.append(Spacer(1, 10))

            story.append(Spacer(1, 25))

        story.append(PageBreak())

    pdf = SimpleDocTemplate(filename)
    pdf.build(story)


# --------------------------------
# Topics (Recent + dynamic)
# --------------------------------
topics = {
    "AI Regulations Worldwide":
        "AI regulation laws worldwide 2025",

    "European AI Act Updates":
        "EU AI Act updates 2025 2026",

    "US AI Policy":
        "United States AI policy regulation 2025",

    "China AI Regulation":
        "China AI regulation policy artificial intelligence",

    "India AI Governance":
        "India artificial intelligence regulation policy",

    "Corporate AI Governance":
        "AI governance frameworks companies 2025"
}


# --------------------------------
# Main
# --------------------------------
if __name__ == "__main__":

    document_sections = {}

    for section, query in topics.items():

        print("Searching:", section)

        results = search_google(query)

        articles = []

        for item in results.get("organic", [])[:10]:

            url = item["url"]

            print("Fetching:", url)

            title, text = extract_article(url)

            if text and len(text) > 2000:
                articles.append((title, text))

        document_sections[section] = articles


    create_pdf("ai_regulation_global_report.pdf", document_sections)

    print("PDF Generated: ai_regulation_global_report.pdf")

Searching: AI Regulations Worldwide
Fetching: https://www.whitehouse.gov/presidential-actions/2025/12/eliminating-state-law-obstruction-of-national-artificial-intelligence-policy/
Fetching: https://www.ncsl.org/technology-and-communication/artificial-intelligence-2025-legislation
Fetching: https://www.cimplifi.com/resources/the-updated-state-of-ai-regulations-for-2025/
Fetching: http://www.mindfoundry.ai/blog/ai-regulations-around-the-world
Fetching: https://iapp.org/news/a/global-ai-law-and-policy-tracker-highlights-and-takeaways
Fetching: https://www.morganlewis.com/pubs/2025/12/the-new-rules-of-ai-a-global-legal-overview
Fetching: https://www.anecdotes.ai/learn/ai-regulations-in-2025-us-eu-uk-japan-china-and-more
Fetching: https://www.whitecase.com/insight-our-thinking/ai-watch-global-regulatory-tracker-united-states
Fetching: https://pubsonline.informs.org/do/10.1287/LYTX.2025.03.10/full/
Searching: European AI Act Updates
Fetching: https://artificialintelligenceact.eu/
Fetching: h

In [2]:
print(item)

{'position': 8, 'title': 'Evolution of Neural Networks to Large Language Models', 'url': 'https://www.labellerr.com/blog/evolution-of-neural-networks-to-large-language-models/', 'destination': 'https://www.labellerr.com › blog › evolution-of-neural-...', 'description': 'Explore the evolution from neural networks to large language models, highlighting key advancements in NLP with the rise of transformer models.Read more', 'isAmp': False}


In [2]:
import requests
import os
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

load_dotenv()

API_KEY = os.getenv("ZENSERP_API_KEY")

# ---------------------------------
# Search match scorecards
# ---------------------------------
def search_matches(query):

    url = "https://app.zenserp.com/api/v2/search"

    params = {
        "q": query,
        "num": 20,
        "gl": "us"
    }

    headers = {"apikey": API_KEY}

    response = requests.get(url, headers=headers, params=params)

    return response.json()


# ---------------------------------
# Extract scorecard + commentary
# ---------------------------------
def scrape_match(url):

    try:

        r = requests.get(url, timeout=10)

        soup = BeautifulSoup(r.text, "html.parser")

        text = soup.get_text()

        title = soup.title.text if soup.title else "Match Report"

        return title, text

    except:
        return None, None


# ---------------------------------
# Create PDF
# ---------------------------------
def create_pdf(filename, matches):

    styles = getSampleStyleSheet()

    story = []

    story.append(Paragraph("Cricket World Cup Detailed Match Reports", styles["Title"]))
    story.append(Spacer(1, 30))

    for title, text in matches:

        story.append(Paragraph(title, styles["Heading2"]))
        story.append(Spacer(1, 15))

        paragraphs = text.split("\n")

        for p in paragraphs:

            if len(p.strip()) > 120:

                story.append(Paragraph(p, styles["BodyText"]))
                story.append(Spacer(1, 8))

        story.append(PageBreak())

    pdf = SimpleDocTemplate(filename)

    pdf.build(story)


# ---------------------------------
# Queries for matches
# ---------------------------------
queries = [

"T20 World Cup 2026 match scorecard commentary",
"India vs Pakistan T20 World Cup commentary full match",
"England vs West Indies T20 World Cup full scorecard",
"New Zealand vs South Africa T20 World Cup match commentary",
"T20 World Cup 2026 full match report commentary"
]


# ---------------------------------
# Main pipeline
# ---------------------------------
if __name__ == "__main__":

    matches = []

    for q in queries:

        print("Searching:", q)

        results = search_matches(q)

        for item in results.get("organic", [])[:8]:

            url = item["url"]

            print("Scraping:", url)

            title, text = scrape_match(url)

            if text and len(text) > 3000:

                matches.append((title, text[:15000]))


    create_pdf("cricket_worldcup_match_reports.pdf", matches)

    print("PDF Generated: cricket_worldcup_match_reports.pdf")

Searching: T20 World Cup 2026 match scorecard commentary
Scraping: https://www.espn.com/cricket/series/8604/game/1512772/india-vs-england-2nd-sf-icc-mens-t20-world-cup-2025-26
Scraping: https://www.cricbuzz.com/live-cricket-scores/139478/ind-vs-eng-2nd-semi-final-icc-mens-t20-world-cup-2026
Scraping: https://www.espncricinfo.com/live-cricket-score
Scraping: https://www.icc-cricket.com/tournaments/mens-t20-world-cup-2026/matches/268128/india-vs-england
Scraping: https://www.cricbuzz.com/cricket-match/live-scores
Scraping: https://www.flashscoreusa.com/cricket/world/t20-world-cup/
Scraping: https://cricheroes.com/tournament/1854614/icc-mens-t20-world-cup-2026/matches/live-matches
Scraping: https://www.olympics.com/en/sports/cricket/live-scores/india-vs-pakistan-match-27-scorecard-2026/match?gamecode=inpk02152026268086
Searching: India vs Pakistan T20 World Cup commentary full match
Scraping: https://www.youtube.com/watch?v=i0r5AIKpneo
Scraping: https://www.bilibili.tv/en/video/4786857934